# Battle Cats Gacha Simulation & Fetch Test
This notebook is used to test the data models and simulate the gacha track logic.

In [1]:
import sys
import os
import pandas as pd
import asyncio
import re
from bs4 import BeautifulSoup

# Add src directory to path so we can import our modules
sys.path.append(os.path.abspath("../src"))

from battle_cat_calc.models import GachaSlot, Position, ParseStatus
from battle_cat_calc.fetcher import BCDataFetcher
from battle_cat_calc.parser import BCParser
from battle_cat_calc.analyzer import SeedAnalyzer
from battle_cat_calc.simulator import GachaSimulator

# 初始化封裝好的 Parser# 初始化封裝好的 Parser

In [2]:
fetcher = BCDataFetcher()
seed = "2127986414"
event = "2025-12-12_1019"

# In Jupyter, we can use await directly in a cell
html = await fetcher.fetch_html(seed=seed, event=event, count=500)
print(f"Successfully fetched {len(html)} characters.")
print("-" * 30)
print(html)

Successfully fetched 1427667 characters.
------------------------------
<!DOCTYPE html>
<html lang="en">
  <head>
    <meta http-equiv="Content-Type" content="text/html; charset=utf-8">
    <meta name="viewport" content="width=device-width, initial-scale=1">
    <meta name="robots" content="none">
    <link rel="icon" type="image/png" href="/asset/image/treasure.png">
    <link href="/asset/style/tacit.css" rel="stylesheet" type="text/css">
    <title>Battle Cats Rolls - Tracks</title>
    <style>
      .rare { background-color: white; }
      .owned { background-color: azure; }
      
      .supa_fest { background-color: yellow; }
      .supa { background-color: gold; }
      .uber_fest { background-color: salmon; }
      .uber { background-color: red; }
      .exclusive { background-color: aqua; }
      .found { background-color: lime; }
      .legend { background-color: darkviolet; }
      .attack { background-color: lavender; }
      .triggered_attack { background-color: pink; }
  

## 4. 使用封裝後的 Parser
現在我們可以使用 `src/battle_cat_calc/parser.py` 中封裝好的 `BCParser` 來一次解析所有資料。

In [3]:
# 初始化封裝好的 Parser
parser = BCParser(html)

# 一次解析所有軌道 (A & B)，現在回傳的是專業的 Track 物件
track_a, track_b = parser.get_tracks()


df_a = pd.DataFrame([
    {
        "No": f"{s.id[0]}{s.id[1]}",
        "Result": s.result,
        "Guaranteed": s.guaranteed,
        "Next Pos": f"{s.next_pos.id}{s.next_pos.track}" if isinstance(s.next_pos, Position) else s.next_pos
    } for s in track_a
])

# 轉換為 DataFrame 檢視 (以 Track B 為例)
df_b = pd.DataFrame([
    {
        "No": f"{s.id[0]}{s.id[1]}",
        "Result": s.result,
        "Guaranteed": s.guaranteed,
        "Next Pos": f"{s.next_pos.id}{s.next_pos.track}" if isinstance(s.next_pos, Position) else s.next_pos
    } for s in track_b
])

df_b.set_index("No", inplace=True)

# 設定 pandas 顯示選項
pd.set_option('display.max_rows', None)

print(f"解析完成: {track_a.name} 軌道有 {len(track_a)} 筆, {track_b.name} 軌道有 {len(track_b)} 筆")

print("Track A:")
display(df_a)
print("Track B:")
display(df_b)


display(pd.DataFrame(parser.get_all_cats()))

解析完成: A 軌道有 500 筆, B 軌道有 500 筆
Track A:


,No,Result,Guaranteed,Next Pos
0,1A,貓咪陰陽師,第10使徒,11B
1,2A,拳擊手貓,白夜姬零,12B
2,3A,搖滾貓,第6使徒,13B
3,4A,神槍手貓,EVA量產機,14B
4,5A,占卜貓,EVA零號機,15B
5,6A,車輪貓,EVA初號機,16B
6,7A,菩薩貓,EVA初號機,17B
7,8A,擊劍貓,第6使徒,18B
8,9A,業餘撐杆跳貓,EVA量產機,19B
9,10A,人魚貓,EVA零號機,20B


Track B:


,Result,Guaranteed,Next Pos
No,,,
1B,貓咪陰陽師,白夜姬零,12B
2B,明日香貓,第6使徒,13B
3B,車輪貓,EVA量產機,14B
4B,薰貓,EVA零號機,15B
5B,御宅貓,EVA初號機,16B
6B,跳跳貓,EVA初號機,17B
7B,神槍手貓,第6使徒,18B
8B,貓咪番長,EVA量產機,19B
9B,拳擊手貓,EVA零號機,20B


,0
0,鬥士貓
1,第10使徒
2,魔劍士貓
3,魔女貓
4,樹剪貓
5,神槍手貓
6,薩滿貓
7,僧侶貓
8,小小明日香
9,窗邊少女貓


In [ ]:
simulator = GachaSimulator(track_a, track_b)

targets = set([
    # "EVA量產機",
    # "EVA零號機",
    # "EVA初號機",
    # "真嗣貓",
    # "白夜姬零",
    # "第6使徒",
    # "第10使徒",
    # "小小零",
    # "小小明日香",  
    # "小小美里",
    "零貓",
    # "明日香貓",
    # "真理貓",
    # "薰貓"
])

result = simulator.find_best_path(Position(1, 'A'), set(targets), 500)

display(result)

DEBUG Pop: 1A Cost=0 Coll=0
DEBUG Pop: 2A Cost=1 Coll=0
DEBUG Pop: 3A Cost=2 Coll=0
DEBUG Pop: 4A Cost=3 Coll=0
DEBUG Pop: 5A Cost=4 Coll=0
DEBUG Pop: 6A Cost=5 Coll=0
DEBUG Pop: 7A Cost=6 Coll=0
DEBUG Pop: 8A Cost=7 Coll=0
DEBUG Pop: 9A Cost=8 Coll=0
DEBUG Pop: 10A Cost=9 Coll=0
DEBUG Pop: 11B Cost=10 Coll=0
DEBUG Pop: 11A Cost=10 Coll=0
DEBUG Pop: 12B Cost=11 Coll=0
DEBUG Pop: 12A Cost=11 Coll=0
DEBUG Pop: 12B Cost=11 Coll=0
DEBUG Pop: 13B Cost=12 Coll=0
DEBUG Pop: 13A Cost=12 Coll=0
DEBUG Pop: 13B Cost=12 Coll=0
DEBUG Pop: 14B Cost=13 Coll=0
DEBUG Pop: 14A Cost=13 Coll=0
DEBUG Pop: 14B Cost=13 Coll=0
DEBUG Pop: 15B Cost=14 Coll=0
DEBUG Pop: 15B Cost=14 Coll=0
DEBUG Pop: 15A Cost=14 Coll=0
DEBUG Pop: 16B Cost=15 Coll=0
DEBUG Pop: 16B Cost=15 Coll=0
DEBUG Pop: 16A Cost=15 Coll=0
DEBUG 16A Unit: '零貓' (Type: <class 'str'>)
DEBUG 16A Unit Ords: [38646, 35987]
DEBUG Pop: 17B Cost=16 Coll=1


{'total_cost': 16,
 'path': (('Single', Position(id=1, track='A'), {'貓咪陰陽師'}),
  ('Single', Position(id=2, track='A'), {'拳擊手貓'}),
  ('Single', Position(id=3, track='A'), {'搖滾貓'}),
  ('Single', Position(id=4, track='A'), {'神槍手貓'}),
  ('Single', Position(id=5, track='A'), {'占卜貓'}),
  ('Single', Position(id=6, track='A'), {'車輪貓'}),
  ('Guaranteed', Position(id=7, track='A'), {'零貓'})),
 'collected': {'零貓'}}